# Fee Doc Scraper

grabs fee-related filings across bankruptcy cases from courtlistener, checks if they're free in recap, pulls the text when they are

need a courtlistener api token (sign-in -> profile -> your api token) saved in a `.env` file:
`COURTLISTENER_TOKEN=your_token_here`

## Setup
add cases to `cases.csv` as you find them

In [1]:
import os
import time
import json
import requests
import re
import pandas as pd
from dotenv import load_dotenv
from collections import defaultdict

load_dotenv()
for folder in ["data/raw", "data/intermediate", "data/final"]:
    os.makedirs(folder, exist_ok=True)


In [2]:
cases_df = pd.read_csv("cases.csv")
CASES = cases_df.to_dict("records")

KEYWORDS = [
    "fee examiner",
    "interim compensation",
    "interim fee application",
    "final fee application",
    "monthly fee statement",
    "fee committee",
    "objection to fee",
    "compensation and reimbursement",
]

TOKEN = os.environ.get("COURTLISTENER_TOKEN", "PASTE_YOUR_TOKEN_HERE")
HEADERS = {"Authorization": f"Token {TOKEN}"}
API_ROOT = "https://www.courtlistener.com/api/rest/v4"

REQUEST_DELAY = 3.5       # keeps us under 20/min
DAILY_REQUEST_LIMIT = 5000 # safety stop
request_count = 0


## The `safe_get` Helper
wraps requests.get so it paces itself and retries if we get rate limited

In [3]:
def safe_get(url, params=None, retries=3, delay=2):
    global request_count

    if request_count >= DAILY_REQUEST_LIMIT:
        print("hit the safety stop, raise DAILY_REQUEST_LIMIT if you need more")
        return None

    for attempt in range(retries):
        try:
            resp = requests.get(url, headers=HEADERS, params=params, timeout=30)
            request_count += 1
            time.sleep(REQUEST_DELAY)
            if resp.status_code == 429:
                wait = delay * (2 ** attempt)
                print(f"rate limited, waiting {wait}s...")
                time.sleep(wait)
                continue
            resp.raise_for_status()
            return resp.json()
        except requests.RequestException as e:
            print(f"request failed ({e}), retry {attempt + 1}/{retries}")
            time.sleep(delay)
    print(f"giving up on {url}")
    return None


## The Scraping Functions
search a docket (server-side filtered so we're not pulling everything), pull out the actual documents, check if they're free + grab text for the high-value ones (fee examiner stuff only, not routine filings)

In [4]:
HIGH_VALUE_TERMS = ["fee examiner", "examiner's report"]

def is_high_value(description):
    d = description.lower()
    return any(term in d for term in HIGH_VALUE_TERMS)


def search_docket_for_fee_documents(docket_number, court, keywords):
    keyword_clause = " OR ".join(f'"{kw}"' for kw in keywords)
    query = f'docketNumber:"{docket_number}" AND court_id:{court} AND ({keyword_clause})'

    results = []
    url = f"{API_ROOT}/search/"
    params = {"q": query, "type": "rd", "page_size": 100}

    while url:
        data = safe_get(url, params=params)
        if not data:
            break
        results.extend(data.get("results", []))
        url = data.get("next")
        params = None

    return results


def extract_documents(search_results):
    # results sometimes come back nested under recap_documents, sometimes flat
    docs = []
    for r in search_results:
        nested = r.get("recap_documents")
        if nested:
            docs.extend(nested)
        else:
            docs.append(r)
    return docs


def process_case(docket_number, court, case_name, keywords):
    print(f"=== {case_name} ({docket_number}, {court}) ===")

    search_results = search_docket_for_fee_documents(docket_number, court, keywords)
    documents = extract_documents(search_results)
    print(f"  {len(documents)} fee-related docs found")

    case_results = []
    available_count = 0

    for doc in documents:
        is_available = doc.get("is_available", False)
        plain_text = doc.get("plain_text")

        if is_available and not plain_text and is_high_value(doc.get("description", "")):
            doc_id = doc.get("id")
            if doc_id:
                full = safe_get(f"{API_ROOT}/recap-documents/{doc_id}/")
                if full:
                    plain_text = full.get("plain_text")

        if plain_text:
            available_count += 1

        case_results.append({
            "case_name": case_name,
            "docket_number": docket_number,
            "court": court,
            "document_id": doc.get("id"),
            "description": doc.get("description", ""),
            "is_available": is_available,
            "plain_text": plain_text,
        })

    print(f"  {available_count} of {len(documents)} had free text\n")
    return case_results


## Run It
loops through everything in CASES, skips anything already scraped, saves progress after each case

In [5]:
if os.path.exists("data/raw/raw_scrape_all_cases.json"):
    with open("data/raw/raw_scrape_all_cases.json") as f:
        all_results = json.load(f)
else:
    all_results = []

already_done = {(r["docket_number"], r["court"]) for r in all_results}

for case in CASES:
    key = (case["docket_number"], case["court"])
    if key in already_done:
        print(f"skipping {case['name']}, already scraped")
        continue
    case_results = process_case(case["docket_number"], case["court"], case["name"], KEYWORDS)
    all_results.extend(case_results)

    with open("data/raw/raw_scrape_all_cases.json", "w", encoding="utf-8") as f:
        json.dump(all_results, f, ensure_ascii=False, indent=2)
    print(f"  saved progress ({len(all_results)} total so far)\n")

print(f"total: {len(all_results)} documents")
print(f"requests used: {request_count}")


skipping Seadrill Limited, already scraped
skipping BlockFi Inc., already scraped
skipping FTX Trading Ltd., already scraped
skipping Purdue Pharma L.P., already scraped
skipping Celsius Network LLC, already scraped
skipping Toys "R" Us, Inc., already scraped
total: 8228 documents
requests used: 0


## Split Out Documents With Real Text

In [6]:
with open("data/raw/raw_scrape_all_cases.json") as f:
    all_results = json.load(f)

with_text = [r for r in all_results if r["plain_text"]]
with open("data/intermediate/with_text_unfiltered.json", "w", encoding="utf-8") as f:
    json.dump(with_text, f, ensure_ascii=False, indent=2)

print(f"{len(with_text)} documents with real text, out of {len(all_results)} total checked")


555 documents with real text, out of 8228 total checked


## Filter To Real Examiner Reports

In [7]:
def is_real_report(description):
    d = description.lower().strip()
    noise_prefixes = [
        "affidavit", "certificate", "certification", "application to employ",
        "application for pro hac vice", "declaration", "order authorizing",
        "notice of appearance", "monthly fee statement",
    ]
    if any(d.startswith(p) for p in noise_prefixes):
        return False
    examiner_pos = d.find("examiner")
    report_pos = d.find("report")
    return examiner_pos != -1 and examiner_pos < 30 and report_pos != -1 and report_pos < 100

examiner_reports = [r for r in with_text if is_real_report(r["description"])]

seen = set()
unique_reports = []
for r in examiner_reports:
    if r['document_id'] not in seen:
        seen.add(r['document_id'])
        unique_reports.append(r)

with open("data/final/fee_examiner_reports_clean.json", "w", encoding="utf-8") as f:
    json.dump(unique_reports, f, ensure_ascii=False, indent=2)

for r in unique_reports:
    print(f"[{r['case_name']}] {r['description'][:80]}")

print(f"\n{len(unique_reports)} unique examiner reports, out of {len(with_text)} total documents with text")


[Seadrill Limited] Notice Fee Examiner's Report Regarding Review of the First Interim Fee Applicati
[Seadrill Limited] Notice FEE EXAMINERS REPORT REGARDING: (I) REVIEW OF THE FINAL FEE APPLICATIONS 
[BlockFi Inc.] Document re: Fee Examiners Corrected Consolidated Final Report Regarding Final F
[BlockFi Inc.] Document re: Fee Examiners Consolidated Final Report Regarding Final Fee Applica
[BlockFi Inc.] Document re: Fee Examiner's Consolidated Final Report Regarding Final Fee Applic
[BlockFi Inc.] Document re: Fee Examiners Consolidated Final Report Regarding Final Fee Applica
[BlockFi Inc.] Document re: Fee Examiner's Consolidated Final Report Regarding Second Interim F
[BlockFi Inc.] Document re: Fee Examiner's Consolidated Final Report Regarding Second Interim F
[BlockFi Inc.] Document re: Fee Examiners Consolidated Final Report Regarding Final Fee Applica
[BlockFi Inc.] Document re: Fee Examiners Consolidated Final Report Regarding First Interim Fee
[BlockFi Inc.] Document re: Fee 

## Quick Check per Case

In [8]:
by_case = defaultdict(lambda: {"total": 0, "with_text": 0})
for r in all_results:
    by_case[r["case_name"]]["total"] += 1
    if r["plain_text"]:
        by_case[r["case_name"]]["with_text"] += 1

for case_name, counts in by_case.items():
    print(f"{case_name}: {counts['with_text']} of {counts['total']} had free text")


Seadrill Limited: 2 of 136 had free text
BlockFi Inc.: 332 of 795 had free text
Purdue Pharma L.P.: 101 of 3369 had free text
Celsius Network LLC: 59 of 1148 had free text
Toys "R" Us, Inc.: 0 of 652 had free text
FTX Trading Ltd.: 61 of 2128 had free text


## Find Real Time-Entry Sections
scans every document with text for a real itemized time-entry section (not just fee examiner reports), isolates just that section

In [9]:
TIME_DETAIL_MARKERS = ["time detail", "detailed time records", "time log", "daily time log"]

def find_time_entry_section(text):
    lower = text.lower()
    for marker in TIME_DETAIL_MARKERS:
        idx = lower.find(marker)
        if idx != -1:
            return text[idx:]
    return None

with open("data/intermediate/with_text_unfiltered.json") as f:
    with_text = json.load(f)

time_entry_docs = []
for r in with_text:
    section = find_time_entry_section(r["plain_text"])
    if section:
        time_entry_docs.append({
            "case_name": r["case_name"],
            "document_id": r["document_id"],
            "description": r["description"],
            "raw_entry_section": section,
        })

with open("data/intermediate/raw_time_entry_sections.json", "w", encoding="utf-8") as f:
    json.dump(time_entry_docs, f, ensure_ascii=False, indent=2)

print(f"{len(time_entry_docs)} documents have a real time-entry section, out of {len(with_text)} with any text")


123 documents have a real time-entry section, out of 555 with any text


## Parse Time Entries
splits on wide column gaps instead of matching name patterns - doesn't depend on capitalization, tested clean against both all-caps and normal documents. anything that doesn't look confident keeps its raw_block instead of guessing, so nothing new gets silently wrong as we add cases

In [10]:
DATE_PATTERN = re.compile(r"^(\d{1,2}[-/]\d{1,2}[-/]\d{2,4})\b")
NUMBER_PATTERN = re.compile(r"\$?\s*([\d,]+\.\d{2})")

def looks_like_a_name(text):
    if not text or len(text) > 40:
        return False
    if any(c.isdigit() for c in text):
        return False
    admin_words = {"invoice", "total", "matter", "page", "client", "balance", "remittance", "desc"}
    if any(w in text.lower() for w in admin_words):
        return False
    return bool(re.fullmatch(r"[A-Za-z.\s]+", text))

def split_into_entry_blocks(text):
    lines = text.split("\n")
    blocks, current = [], []
    for line in lines:
        stripped = line.rstrip()
        if not stripped.strip():
            continue
        if DATE_PATTERN.match(stripped.strip()):
            if current:
                blocks.append("\n".join(current))
            current = [stripped.strip()]
        elif current:
            current.append(stripped.strip())
    if current:
        blocks.append("\n".join(current))
    return blocks

def parse_entry_block(block):
    date_match = DATE_PATTERN.match(block)
    date = date_match.group(1) if date_match else None
    remainder = block[date_match.end():] if date_match else block

    columns = [c.strip() for c in re.split(r"\s{2,}", remainder) if c.strip()]
    raw_timekeeper = columns[0] if columns else None
    rest = " ".join(columns[1:]) if len(columns) > 1 else ""

    numbers = [float(n.replace(",", "")) for n in NUMBER_PATTERN.findall(rest)]
    hours = next((n for n in numbers if 0 < n <= 24), None)

    narrative = re.sub(r"\$?\s*[\d,]+\.\d{2}", "", rest)
    narrative = re.sub(r"\[.*?\]", "", narrative)
    narrative = " ".join(narrative.split()).strip()

    name_ok = raw_timekeeper is not None and looks_like_a_name(raw_timekeeper)
    high_confidence = bool(date and name_ok and hours is not None and len(narrative) > 5)

    return {
        "date": date,
        "timekeeper": raw_timekeeper if name_ok else None,
        "hours": hours,
        "narrative": narrative if high_confidence else None,
        "high_confidence": high_confidence,
        "raw_block": block,
    }

with open("data/intermediate/raw_time_entry_sections.json") as f:
    time_entry_docs = json.load(f)

all_entries = []
for doc in time_entry_docs:
    for block in split_into_entry_blocks(doc["raw_entry_section"]):
        parsed = parse_entry_block(block)
        parsed["case_name"] = doc["case_name"]
        parsed["document_id"] = doc["document_id"]
        all_entries.append(parsed)

df = pd.DataFrame(all_entries)
df.to_csv("data/final/parsed_time_entries.csv", index=False)

high_conf = df["high_confidence"].sum()
print(f"{len(df)} total entries, {high_conf} high-confidence ({high_conf/len(df)*100:.0f}%)")


30998 total entries, 18061 high-confidence (58%)
